In [1]:
# === 非等间隔时间序列 → 等间隔时间序列（按小时重采样 + 插值）===
import pandas as pd

def preprocess_irregular_series(csv_path, freq='H'):
    """
    输入 CSV 文件路径，返回 DataFrame：已重采样 + 线性插值
    参数 freq：如 'H' 表示每小时；'D' 表示每天
    """
    df = pd.read_csv(csv_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp').sort_index()
    
    df_resampled = df.resample(freq).mean()
    df_interpolated = df_resampled.interpolate(method='linear')
    
    return df_interpolated.reset_index()

# 使用示例
file_path = "'./data/data_clean/productivity.csv'"  # 替换为你的文件名
df_processed = preprocess_irregular_series(file_path)
df_processed.to_csv("productivity_processed.csv", index=False)


/var/folders/25/xxhslgk16kq33dxhh4440t9r0000gn/T/ipykernel_2579/4131540452.py:13: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_resampled = df.resample(freq).mean()


# Air_quality

In [ ]:
import os
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from darts import TimeSeries
from darts.models import TransformerModel
from darts.metrics import mae, mape
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from pytorch_lightning.loggers import CSVLogger

class UnifiedTransformerForecaster:
    def __init__(self, model_params: dict = None, default_config: dict = None):
        self.base_params = {
            'input_chunk_length': 36,
            'output_chunk_length': 12,
            'd_model': 64,
            'nhead': 4,
            'num_encoder_layers': 2,
            'num_decoder_layers': 2,
            'batch_size': 16,
            'n_epochs': 50,
            'pl_trainer_kwargs': {'accelerator': 'cpu', 'enable_progress_bar': False}
        }
        if model_params:
            self.base_params.update(model_params)

        # 提前定义 config
        self.config = {
            'results_dir': './forecast_results',
            'figure_dpi': 300,
            'train_ratio': 0.7,
            'val_ratio': 0.15
        }
        if default_config:
            self.config.update(default_config)

        # 设置 Logger
        from pytorch_lightning.loggers import CSVLogger
        self.base_params['pl_trainer_kwargs']['logger'] = CSVLogger(save_dir=self.config['results_dir'])

        # 初始化模型
        self.model = TransformerModel(**self.base_params)

        os.makedirs(self.config['results_dir'], exist_ok=True)

    def load_dataset(self, file_path: str, freq: str = None):
        df = pd.read_csv(file_path, encoding='utf-8', sep=',')
        dataset_name = os.path.basename(file_path).split('.')[0]
        # ———— 去重聚合：对相同 timestamp 取 y 的均值 ————
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = (
            df.sort_values('timestamp')
              .groupby('timestamp', as_index=False)['y']
              .mean()
        )

        # 生成完备索引并重建 DataFrame
        full_idx = self._generate_complete_index(df['timestamp'],freq)
        df = df.set_index('timestamp').reindex(full_idx).reset_index().rename(columns={'index':'timestamp'})
        df['y'] = df['y'].ffill()

        series = TimeSeries.from_dataframe(
            df,
            time_col='timestamp',
            value_cols='y',
            freq=freq or pd.infer_freq(full_idx) or 'h'
        )
        return dataset_name, series


    def _generate_complete_index(self, original_index, freq: str=None):
        """
        生成从 min 到 max 的完整时间索引。
        - 若提供 freq_override，则直接用它；
        - 否则尝试 pd.infer_freq 推断；
        - 推断失败时用最小正差值的 offset。
        """
        if len(original_index) < 2:
            return original_index
        
        if freq:
            freq_str = freq
        else:
            freq_str = pd.infer_freq(original_index) or ''
            if not freq_str:
                diffs    = pd.Series(original_index).diff().dropna()
                min_diff = diffs[diffs > pd.Timedelta(0)].min()
                freq_str = pd.tseries.frequencies.to_offset(min_diff).freqstr
       
        return pd.date_range(
            start=original_index.min(),
            end=original_index.max(),
            freq=freq_str
        )

    def train_and_predict(self, series: TimeSeries, dataset_name: str):
        try:
            total_len = len(series)
            train_size = int(total_len * self.config['train_ratio'])
            val_size = int(total_len * self.config['val_ratio'])
            train = series[:train_size]
            val = series[train_size:train_size + val_size]
            test = series[train_size + val_size:]

            print(f"\n📊 数据集划分:")
            print(f"训练集: {len(train)} 个点")
            print(f"验证集: {len(val)} 个点")
            print(f"测试集: {len(test)} 个点")

            print("\n🚀 开始训练模型...")
            start_time = time.time()
            self.model.fit(train, val_series=val, verbose=True)

            forecast = None
            history = train.append(val)

            for i in range(0, len(test), self.base_params['output_chunk_length']):
                current_input = history[-self.base_params['input_chunk_length']:]
                n_out = min(self.base_params['output_chunk_length'], len(test) - i)
                pred = self.model.predict(n=n_out, series=current_input)
                if pred is None or pred.values().size == 0:
                    print(f"❌ 第 {i} 步预测失败，终止滚动预测。")
                    break
                forecast = pred if forecast is None else forecast.append(pred)
                if i + self.base_params['output_chunk_length'] <= len(test):
                    history = history.append(test[i:i+self.base_params['output_chunk_length']])

            if forecast is None:
                print(f"❌ {dataset_name} 所有预测失败，跳过保存与绘图。")
                return

            runtime = time.time() - start_time
            self._save_results(train, test, forecast, dataset_name)

            y_true = test.values().flatten()
            y_pred = forecast.values().flatten()
            metrics = self.evaluate_all_metrics(y_true, y_pred, runtime=runtime)
            metrics_path = os.path.join("results/metric_csv/transformer", f"{dataset_name}_metrics.csv")
            pd. DataFrame([metrics]).to_csv(metrics_path, index=False)
            print(f"📄 指标结果已保存至: {metrics_path}")
            print("\n📈 评估指标:")
            for k, v in metrics.items():
                print(f"{k}: {v:.4f}")
            self._plot_comparison(train, test, forecast, dataset_name)
            self._plot_loss_curve(dataset_name)

        except Exception as e:
            print(f"处理数据集 {dataset_name} 时出错: {str(e)}")

    def _plot_loss_curve(self, dataset_name):
        try:
            log_dir = self.model.trainer.logger.log_dir if self.model.trainer.logger else None
            if log_dir is None:
                print("⚠️ 无法获取 logger 日志目录")
                return

            log_path = os.path.join(log_dir, "metrics.csv")
            if not os.path.exists(log_path):
                print("⚠️ 未找到日志文件，无法绘制损失图")
                return

            df = pd.read_csv(log_path)

            # 检查是否存在列
            if 'epoch' not in df.columns:
                print("⚠️ metrics.csv 缺少 epoch 列")
                return

            train_loss = df[df['train_loss'].notna()][['epoch', 'train_loss']] if 'train_loss' in df.columns else pd.DataFrame()
            val_loss = df[df['val_loss'].notna()][['epoch', 'val_loss']] if 'val_loss' in df.columns else pd.DataFrame()

            plt.figure()
            if not train_loss.empty:
                plt.plot(train_loss['epoch'], train_loss['train_loss'], label="train_loss")
            if not val_loss.empty:
                plt.plot(val_loss['epoch'], val_loss['val_loss'], label="val_loss")

            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.legend()
            plt.title(f"{dataset_name} Loss Curve")
            plt.tight_layout()
            plt.savefig(os.path.join("results/loss_png/transformer", f"{dataset_name}_loss_curve.png"), dpi=self.config['figure_dpi'])
            plt.close()
        except Exception as e:
            print(f"绘图失败: {e}")

    def mean_absolute_percentage_error(self, y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

    def symmetric_mean_absolute_percentage_error(self, y_true, y_pred):
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10))

    def directional_accuracy(self, y_true, y_pred):
        return np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))

    def threshold_accuracy(self, y_true, y_pred, threshold=100):
        return accuracy_score((y_true > threshold).astype(int), (y_pred > threshold).astype(int))

    def evaluate_all_metrics(self, y_true, y_pred, threshold=100, runtime=None):
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAPE': self.mean_absolute_percentage_error(y_true, y_pred),
            'sMAPE': self.symmetric_mean_absolute_percentage_error(y_true, y_pred),
            'Directional_Accuracy': self.directional_accuracy(y_true, y_pred),
            f'Threshold_Accuracy(>{threshold})': self.threshold_accuracy(y_true, y_pred, threshold),
            'Runtime_Seconds': runtime if runtime is not None else -1
        }

    def _save_results(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        try:
            train_df = train.to_dataframe().rename(columns={train.components[0]: 'value'})
            test_df = test.to_dataframe().rename(columns={test.components[0]: 'value'})
            forecast_df = forecast.to_dataframe().rename(columns={forecast.components[0]: 'value'})
            train_df['type'] = 'train'
            test_df['type'] = 'test'
            forecast_df['type'] = 'forecast'
            result_df = pd.concat([train_df, test_df, forecast_df])
            if 'timestamp' not in result_df.columns:
                result_df = result_df.reset_index().rename(columns={'index': 'timestamp'})
            save_path = os.path.join(self.config['results_dir'], f"{dataset_name}_forecast.csv")
            result_df.to_csv(save_path, index=False)
            print(f"✅ {dataset_name} 结果已保存至: {save_path}")
        except Exception as e:
            print(f"保存结果时出错: {str(e)}")

    def _plot_comparison(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        plt.figure(figsize=(14, 7))
        test.plot(label="test", color='green', alpha=0.7)
        for i in range(0, len(forecast), self.base_params['output_chunk_length']):
            chunk = forecast[i:i+self.base_params['output_chunk_length']]
            chunk.plot(
                label="predict" if i == 0 else None,
                color='red',
                linestyle='--',
                marker='.',
                markersize=5
            )
        mae_score = mae(test, forecast)
        mape_score = mape(test, forecast)
        plt.title(f"{dataset_name} (MAE={mae_score:.2f}, MAPE={mape_score:.2f}%)")
        plt.axvline(test.start_time(), color='gray', linestyle='--', alpha=0.5)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        fig_path = os.path.join("results/prediction_png/transformer", f"{dataset_name}_comparison.png")

        plt.savefig(fig_path, dpi=self.config['figure_dpi'])
        plt.close()
        print(f"📊 {dataset_name} 可视化结果已保存")

if __name__ == "__main__":
    DATASET_PATHS = [
        './data/data_clean/air_quality.csv'
    ]
    MODEL_CONFIG = {
        'input_chunk_length': 36,
        'output_chunk_length': 12,
        'd_model': 64,
        'n_epochs': 50
    }
    forecaster = UnifiedTransformerForecaster(model_params=MODEL_CONFIG)
    for data_path in DATASET_PATHS:
        name, series = forecaster.load_dataset(data_path,freq='H')
        print(f"\n🔍 开始处理数据集: {name}")
        print(f"数据长度: {len(series)}")
        print(f"时间频率: {series.freq_str}")
        forecaster.train_and_predict(series, name)




/var/folders/25/xxhslgk16kq33dxhh4440t9r0000gn/T/ipykernel_19679/2981417278.py:92: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/darts/timeseries.py:5248: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  resampled_time_index = resampled_time_index.asfreq(freq)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightnin


🔍 开始处理数据集: air_quality
数据长度: 9357
时间频率: h

📊 数据集划分:
训练集: 6549 个点
验证集: 1403 个点
测试集: 1405 个点

🚀 开始训练模型...
Epoch 49: 100%|██████████| 407/407 [00:08<00:00, 49.16it/s, v_num=3, train_loss=1.510, val_loss=1.400]

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 407/407 [00:08<00:00, 49.15it/s, v_num=3, train_loss=1.510, val_loss=1.400]


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU ava

✅ air_quality 结果已保存至: ./forecast_results/air_quality_forecast.csv

📈 评估指标:
MAE: 0.6537
RMSE: 0.9086
MAPE: 54.4117
sMAPE: 34.3936
Directional_Accuracy: 0.6353
Threshold_Accuracy(>100): 1.0000
Runtime_Seconds: 451.2193


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/darts/timeseries.py:4604: UserWarning: marker is redundantly defined by the 'marker' keyword argument and the fmt string "o" (-> marker='o'). The keyword argument will take precedence.
  p = ax.plot(


📊 air_quality 可视化结果已保存


# Energy

In [ ]:
import os
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from darts import TimeSeries
from darts.models import TransformerModel
from darts.metrics import mae, mape
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from pytorch_lightning.loggers import CSVLogger

class UnifiedTransformerForecaster:
    def __init__(self, model_params: dict = None, default_config: dict = None):
        self.base_params = {
            'input_chunk_length': 36,
            'output_chunk_length': 12,
            'd_model': 64,
            'nhead': 4,
            'num_encoder_layers': 2,
            'num_decoder_layers': 2,
            'batch_size': 16,
            'n_epochs': 50,
            'pl_trainer_kwargs': {'accelerator': 'cpu', 'enable_progress_bar': False}
        }
        if model_params:
            self.base_params.update(model_params)

        # 提前定义 config
        self.config = {
            'results_dir': './forecast_results',
            'figure_dpi': 300,
            'train_ratio': 0.7,
            'val_ratio': 0.15
        }
        if default_config:
            self.config.update(default_config)

        # 设置 Logger
        from pytorch_lightning.loggers import CSVLogger
        self.base_params['pl_trainer_kwargs']['logger'] = CSVLogger(save_dir=self.config['results_dir'])

        # 初始化模型
        self.model = TransformerModel(**self.base_params)

        os.makedirs(self.config['results_dir'], exist_ok=True)

    def load_dataset(self, file_path: str, freq: str = None):
        df = pd.read_csv(file_path, encoding='utf-8', sep=',')
        dataset_name = os.path.basename(file_path).split('.')[0]
        # ———— 去重聚合：对相同 timestamp 取 y 的均值 ————
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = (
            df.sort_values('timestamp')
              .groupby('timestamp', as_index=False)['y']
              .mean()
        )

        # 生成完备索引并重建 DataFrame
        full_idx = self._generate_complete_index(df['timestamp'],freq)
        df = df.set_index('timestamp').reindex(full_idx).reset_index().rename(columns={'index':'timestamp'})
        df['y'] = df['y'].ffill()

        series = TimeSeries.from_dataframe(
            df,
            time_col='timestamp',
            value_cols='y',
            freq=freq or pd.infer_freq(full_idx) or 'h'
        )
        return dataset_name, series


    def _generate_complete_index(self, original_index, freq: str=None):
        """
        生成从 min 到 max 的完整时间索引。
        - 若提供 freq_override，则直接用它；
        - 否则尝试 pd.infer_freq 推断；
        - 推断失败时用最小正差值的 offset。
        """
        if len(original_index) < 2:
            return original_index
        
        if freq:
            freq_str = freq
        else:
            freq_str = pd.infer_freq(original_index) or ''
            if not freq_str:
                diffs    = pd.Series(original_index).diff().dropna()
                min_diff = diffs[diffs > pd.Timedelta(0)].min()
                freq_str = pd.tseries.frequencies.to_offset(min_diff).freqstr
       
        return pd.date_range(
            start=original_index.min(),
            end=original_index.max(),
            freq=freq_str
        )

    def train_and_predict(self, series: TimeSeries, dataset_name: str):
        try:
            total_len = len(series)
            train_size = int(total_len * self.config['train_ratio'])
            val_size = int(total_len * self.config['val_ratio'])
            train = series[:train_size]
            val = series[train_size:train_size + val_size]
            test = series[train_size + val_size:]

            print(f"\n📊 数据集划分:")
            print(f"训练集: {len(train)} 个点")
            print(f"验证集: {len(val)} 个点")
            print(f"测试集: {len(test)} 个点")

            print("\n🚀 开始训练模型...")
            start_time = time.time()
            self.model.fit(train, val_series=val, verbose=True)

            forecast = None
            history = train.append(val)

            for i in range(0, len(test), self.base_params['output_chunk_length']):
                current_input = history[-self.base_params['input_chunk_length']:]
                n_out = min(self.base_params['output_chunk_length'], len(test) - i)
                pred = self.model.predict(n=n_out, series=current_input)
                if pred is None or pred.values().size == 0:
                    print(f"❌ 第 {i} 步预测失败，终止滚动预测。")
                    break
                forecast = pred if forecast is None else forecast.append(pred)
                if i + self.base_params['output_chunk_length'] <= len(test):
                    history = history.append(test[i:i+self.base_params['output_chunk_length']])

            if forecast is None:
                print(f"❌ {dataset_name} 所有预测失败，跳过保存与绘图。")
                return

            runtime = time.time() - start_time
            self._save_results(train, test, forecast, dataset_name)

            y_true = test.values().flatten()
            y_pred = forecast.values().flatten()
            metrics = self.evaluate_all_metrics(y_true, y_pred, runtime=runtime)
            metrics_path = os.path.join("results/metric_csv/transformer", f"{dataset_name}_metrics.csv")
            pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
            print(f"📄 指标结果已保存至: {metrics_path}")
            print("\n📈 评估指标:")
            for k, v in metrics.items():
                print(f"{k}: {v:.4f}")

            self._plot_comparison(train, test, forecast, dataset_name)
            self._plot_loss_curve(dataset_name)

        except Exception as e:
            print(f"处理数据集 {dataset_name} 时出错: {str(e)}")

    def _plot_loss_curve(self, dataset_name):
        try:
            log_dir = self.model.trainer.logger.log_dir if self.model.trainer.logger else None
            if log_dir is None:
                print("⚠️ 无法获取 logger 日志目录")
                return

            log_path = os.path.join(log_dir, "metrics.csv")
            if not os.path.exists(log_path):
                print("⚠️ 未找到日志文件，无法绘制损失图")
                return

            df = pd.read_csv(log_path)

            # 检查是否存在列
            if 'epoch' not in df.columns:
                print("⚠️ metrics.csv 缺少 epoch 列")
                return

            train_loss = df[df['train_loss'].notna()][['epoch', 'train_loss']] if 'train_loss' in df.columns else pd.DataFrame()
            val_loss = df[df['val_loss'].notna()][['epoch', 'val_loss']] if 'val_loss' in df.columns else pd.DataFrame()

            plt.figure()
            if not train_loss.empty:
                plt.plot(train_loss['epoch'], train_loss['train_loss'], label="train_loss")
            if not val_loss.empty:
                plt.plot(val_loss['epoch'], val_loss['val_loss'], label="val_loss")

            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.legend()
            plt.title(f"{dataset_name} Loss Curve")
            plt.tight_layout()
            fig_path = os.path.join("results/prediction_png/transformer", f"{dataset_name}_comparison.png")

            plt.close()
        except Exception as e:
            print(f"绘图失败: {e}")

    def mean_absolute_percentage_error(self, y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

    def symmetric_mean_absolute_percentage_error(self, y_true, y_pred):
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10))

    def directional_accuracy(self, y_true, y_pred):
        return np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))

    def threshold_accuracy(self, y_true, y_pred, threshold=100):
        return accuracy_score((y_true > threshold).astype(int), (y_pred > threshold).astype(int))

    def evaluate_all_metrics(self, y_true, y_pred, threshold=100, runtime=None):
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAPE': self.mean_absolute_percentage_error(y_true, y_pred),
            'sMAPE': self.symmetric_mean_absolute_percentage_error(y_true, y_pred),
            'Directional_Accuracy': self.directional_accuracy(y_true, y_pred),
            f'Threshold_Accuracy(>{threshold})': self.threshold_accuracy(y_true, y_pred, threshold),
            'Runtime_Seconds': runtime if runtime is not None else -1
        }

    def _save_results(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        try:
            train_df = train.to_dataframe().rename(columns={train.components[0]: 'value'})
            test_df = test.to_dataframe().rename(columns={test.components[0]: 'value'})
            forecast_df = forecast.to_dataframe().rename(columns={forecast.components[0]: 'value'})
            train_df['type'] = 'train'
            test_df['type'] = 'test'
            forecast_df['type'] = 'forecast'
            result_df = pd.concat([train_df, test_df, forecast_df])
            if 'timestamp' not in result_df.columns:
                result_df = result_df.reset_index().rename(columns={'index': 'timestamp'})
            save_path = os.path.join(self.config['results_dir'], f"{dataset_name}_forecast.csv")
            result_df.to_csv(save_path, index=False)
            print(f"✅ {dataset_name} 结果已保存至: {save_path}")
        except Exception as e:
            print(f"保存结果时出错: {str(e)}")

    def _plot_comparison(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        plt.figure(figsize=(14, 7))
        test.plot(label="test", color='green', alpha=0.7)
        for i in range(0, len(forecast), self.base_params['output_chunk_length']):
            chunk = forecast[i:i+self.base_params['output_chunk_length']]
            chunk.plot(
                label="predict" if i == 0 else None,
                color='red',
                linestyle='--',
                marker='.',
                markersize=5
            )
        mae_score = mae(test, forecast)
        mape_score = mape(test, forecast)
        plt.title(f"{dataset_name} (MAE={mae_score:.2f}, MAPE={mape_score:.2f}%)")
        plt.axvline(test.start_time(), color='gray', linestyle='--', alpha=0.5)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        fig_path = os.path.join("results/prediction_png/transformer", f"{dataset_name}_comparison.png")


        plt.savefig(fig_path, dpi=self.config['figure_dpi'])
        plt.close()
        print(f"📊 {dataset_name} 可视化结果已保存")

if __name__ == "__main__":
    DATASET_PATHS = [
        './data/data_clean/energy.csv'
    ]
    MODEL_CONFIG = {
        'input_chunk_length': 36,
        'output_chunk_length': 12,
        'd_model': 64,
        'n_epochs': 50
    }
    forecaster = UnifiedTransformerForecaster(model_params=MODEL_CONFIG)
    for data_path in DATASET_PATHS:
        name, series = forecaster.load_dataset(data_path,freq='10min')
        print(f"\n🔍 开始处理数据集: {name}")
        print(f"数据长度: {len(series)}")
        print(f"时间频率: {series.freq_str}")
        forecaster.train_and_predict(series, name)




/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.

  | Name                | Type                | Params | Mode 
--------------------------------------------------------------------
0 | criterion           | MSELoss             | 0      | train
1 | train_criterion     | MSELoss             | 0      | train
2 | val_criterion       | MSELoss             | 0      | train
3 | train_metrics       | MetricCollecti


🔍 开始处理数据集: energy
数据长度: 19735
时间频率: 10min

📊 数据集划分:
训练集: 13814 个点
验证集: 2960 个点
测试集: 2961 个点

🚀 开始训练模型...
Epoch 49: 100%|██████████| 861/861 [00:17<00:00, 48.25it/s, v_num=4, train_loss=1.95e+4, val_loss=8.2e+3] 

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 861/861 [00:17<00:00, 48.25it/s, v_num=4, train_loss=1.95e+4, val_loss=8.2e+3]


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU ava

✅ energy 结果已保存至: ./forecast_results/energy_forecast.csv

📈 评估指标:
MAE: 52.5435
RMSE: 90.9101
MAPE: 58.3117
sMAPE: 47.3877
Directional_Accuracy: 0.3449
Threshold_Accuracy(>100): 0.7876
Runtime_Seconds: 891.8806
📊 energy 可视化结果已保存


# gait

In [12]:
import os
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from darts import TimeSeries
from darts.models import TransformerModel
from darts.metrics import mae, mape
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from pytorch_lightning.loggers import CSVLogger

class UnifiedTransformerForecaster:
    def __init__(self, model_params: dict = None, default_config: dict = None):
        self.base_params = {
            'input_chunk_length': 36,
            'output_chunk_length': 12,
            'd_model': 64,
            'nhead': 4,
            'num_encoder_layers': 2,
            'num_decoder_layers': 2,
            'batch_size': 16,
            'n_epochs': 50,
            'pl_trainer_kwargs': {'accelerator': 'cpu', 'enable_progress_bar': False}
        }
        if model_params:
            self.base_params.update(model_params)

        # 提前定义 config
        self.config = {
            'results_dir': './forecast_results',
            'figure_dpi': 300,
            'train_ratio': 0.6,
            'val_ratio': 0.25
        }
        if default_config:
            self.config.update(default_config)

        # 设置 Logger
        from pytorch_lightning.loggers import CSVLogger
        self.base_params['pl_trainer_kwargs']['logger'] = CSVLogger(save_dir=self.config['results_dir'])

        # 初始化模型
        self.model = TransformerModel(**self.base_params)

        os.makedirs(self.config['results_dir'], exist_ok=True)

    def load_dataset(self, file_path: str, freq: str = None):
        df = pd.read_csv(file_path, encoding='utf-8', sep=',')
        dataset_name = os.path.basename(file_path).split('.')[0]
        # ———— 去重聚合：对相同 timestamp 取 y 的均值 ————
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = (
            df.sort_values('timestamp')
              .groupby('timestamp', as_index=False)['y']
              .mean()
        )

        # 生成完备索引并重建 DataFrame
        full_idx = self._generate_complete_index(df['timestamp'],freq)
        df = df.set_index('timestamp').reindex(full_idx).reset_index().rename(columns={'index':'timestamp'})
        df['y'] = df['y'].ffill()

        series = TimeSeries.from_dataframe(
            df,
            time_col='timestamp',
            value_cols='y',
            freq=freq or pd.infer_freq(full_idx) or 'h'
        )
        return dataset_name, series


    def _generate_complete_index(self, original_index, freq: str=None):
        """
        生成从 min 到 max 的完整时间索引。
        - 若提供 freq_override，则直接用它；
        - 否则尝试 pd.infer_freq 推断；
        - 推断失败时用最小正差值的 offset。
        """
        if len(original_index) < 2:
            return original_index
        
        if freq:
            freq_str = freq
        else:
            freq_str = pd.infer_freq(original_index) or ''
            if not freq_str:
                diffs    = pd.Series(original_index).diff().dropna()
                min_diff = diffs[diffs > pd.Timedelta(0)].min()
                freq_str = pd.tseries.frequencies.to_offset(min_diff).freqstr
       
        return pd.date_range(
            start=original_index.min(),
            end=original_index.max(),
            freq=freq_str
        )

    def train_and_predict(self, series: TimeSeries, dataset_name: str):
        try:
            total_len = len(series)
            train_size = int(total_len * self.config['train_ratio'])
            val_size = int(total_len * self.config['val_ratio'])
            train = series[:train_size]
            val = series[train_size:train_size + val_size]
            test = series[train_size + val_size:]

            print(f"\n📊 数据集划分:")
            print(f"训练集: {len(train)} 个点")
            print(f"验证集: {len(val)} 个点")
            print(f"测试集: {len(test)} 个点")

            print("\n🚀 开始训练模型...")
            start_time = time.time()
            self.model.fit(train, val_series=val, verbose=True)

            forecast = None
            history = train.append(val)

            for i in range(0, len(test), self.base_params['output_chunk_length']):
                current_input = history[-self.base_params['input_chunk_length']:]
                n_out = min(self.base_params['output_chunk_length'], len(test) - i)
                pred = self.model.predict(n=n_out, series=current_input)
                if pred is None or pred.values().size == 0:
                    print(f"❌ 第 {i} 步预测失败，终止滚动预测。")
                    break
                forecast = pred if forecast is None else forecast.append(pred)
                if i + self.base_params['output_chunk_length'] <= len(test):
                    history = history.append(test[i:i+self.base_params['output_chunk_length']])

            if forecast is None:
                print(f"❌ {dataset_name} 所有预测失败，跳过保存与绘图。")
                return

            runtime = time.time() - start_time
            self._save_results(train, test, forecast, dataset_name)

            y_true = test.values().flatten()
            y_pred = forecast.values().flatten()
            metrics = self.evaluate_all_metrics(y_true, y_pred, runtime=runtime)
            metrics_path = os.path.join("results/metric_csv/transformer", f"{dataset_name}_metrics.csv")
            pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
            print(f"📄 指标结果已保存至: {metrics_path}")
            print("\n📈 评估指标:")
            for k, v in metrics.items():
                print(f"{k}: {v:.4f}")

            self._plot_comparison(train, test, forecast, dataset_name)
            self._plot_loss_curve(dataset_name)

        except Exception as e:
            print(f"处理数据集 {dataset_name} 时出错: {str(e)}")

    def _plot_loss_curve(self, dataset_name):
        try:
            log_dir = self.model.trainer.logger.log_dir if self.model.trainer.logger else None
            if log_dir is None:
                print("⚠️ 无法获取 logger 日志目录")
                return

            log_path = os.path.join(log_dir, "metrics.csv")
            if not os.path.exists(log_path):
                print("⚠️ 未找到日志文件，无法绘制损失图")
                return

            df = pd.read_csv(log_path)

            # 检查是否存在列
            if 'epoch' not in df.columns:
                print("⚠️ metrics.csv 缺少 epoch 列")
                return

            train_loss = df[df['train_loss'].notna()][['epoch', 'train_loss']] if 'train_loss' in df.columns else pd.DataFrame()
            val_loss = df[df['val_loss'].notna()][['epoch', 'val_loss']] if 'val_loss' in df.columns else pd.DataFrame()

            plt.figure()
            if not train_loss.empty:
                plt.plot(train_loss['epoch'], train_loss['train_loss'], label="train_loss")
            if not val_loss.empty:
                plt.plot(val_loss['epoch'], val_loss['val_loss'], label="val_loss")

            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.legend()
            plt.title(f"{dataset_name} Loss Curve")
            plt.tight_layout()
            plt.savefig(os.path.join("results/loss_png/transformer", f"{dataset_name}_loss_curve.png"), dpi=self.config['figure_dpi'])

            plt.close()
        except Exception as e:
            print(f"绘图失败: {e}")

    def mean_absolute_percentage_error(self, y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

    def symmetric_mean_absolute_percentage_error(self, y_true, y_pred):
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10))

    def directional_accuracy(self, y_true, y_pred):
        return np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))

    def threshold_accuracy(self, y_true, y_pred, threshold=100):
        return accuracy_score((y_true > threshold).astype(int), (y_pred > threshold).astype(int))

    def evaluate_all_metrics(self, y_true, y_pred, threshold=100, runtime=None):
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAPE': self.mean_absolute_percentage_error(y_true, y_pred),
            'sMAPE': self.symmetric_mean_absolute_percentage_error(y_true, y_pred),
            'Directional_Accuracy': self.directional_accuracy(y_true, y_pred),
            f'Threshold_Accuracy(>{threshold})': self.threshold_accuracy(y_true, y_pred, threshold),
            'Runtime_Seconds': runtime if runtime is not None else -1
        }

    def _save_results(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        try:
            train_df = train.to_dataframe().rename(columns={train.components[0]: 'value'})
            test_df = test.to_dataframe().rename(columns={test.components[0]: 'value'})
            forecast_df = forecast.to_dataframe().rename(columns={forecast.components[0]: 'value'})
            train_df['type'] = 'train'
            test_df['type'] = 'test'
            forecast_df['type'] = 'forecast'
            result_df = pd.concat([train_df, test_df, forecast_df])
            if 'timestamp' not in result_df.columns:
                result_df = result_df.reset_index().rename(columns={'index': 'timestamp'})
            save_path = os.path.join(self.config['results_dir'], f"{dataset_name}_forecast.csv")
            result_df.to_csv(save_path, index=False)
            print(f"✅ {dataset_name} 结果已保存至: {save_path}")
        except Exception as e:
            print(f"保存结果时出错: {str(e)}")

    def _plot_comparison(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        plt.figure(figsize=(14, 7))
        test.plot(label="test", color='green', alpha=0.7)
        for i in range(0, len(forecast), self.base_params['output_chunk_length']):
            chunk = forecast[i:i+self.base_params['output_chunk_length']]
            chunk.plot(
                label="predict" if i == 0 else None,
                color='red',
                linestyle='--',
                marker='.',
                markersize=5
            )
        mae_score = mae(test, forecast)
        mape_score = mape(test, forecast)
        plt.title(f"{dataset_name} (MAE={mae_score:.2f}, MAPE={mape_score:.2f}%)")
        plt.axvline(test.start_time(), color='gray', linestyle='--', alpha=0.5)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        fig_path = os.path.join("results/prediction_png/transformer", f"{dataset_name}_comparison.png")

        plt.savefig(fig_path, dpi=self.config['figure_dpi'])
        plt.close()
        print(f"📊 {dataset_name} 可视化结果已保存")

if __name__ == "__main__":
    DATASET_PATHS = [
       './data/data_clean/gait.csv'
    ]
    MODEL_CONFIG = {
        'input_chunk_length': 36,
        'output_chunk_length': 12,
        'd_model': 64,
        'n_epochs': 40
    }
    forecaster = UnifiedTransformerForecaster(model_params=MODEL_CONFIG)
    for data_path in DATASET_PATHS:
        name, series = forecaster.load_dataset(data_path,freq='10ms')
        print(f"\n🔍 开始处理数据集: {name}")
        print(f"数据长度: {len(series)}")
        print(f"时间频率: {series.freq_str}")
        forecaster.train_and_predict(series, name)




Epoch 0:  12%|█▏        | 831/6815 [00:24<02:57, 33.77it/s, v_num=11, train_loss=67.70]

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.

  | Name                | Type                | Params | Mode 
--------------------------------------------------------------------
0 | criterion           | MSELoss             | 0      | train
1 | train_criterion     | MSELoss             | 0      | train
2 | val_criterion       | MSELoss             | 0      | train
3 | train_metrics       | MetricCollecti



🔍 开始处理数据集: gait
数据长度: 181800
时间频率: 10ms

📊 数据集划分:
训练集: 109080 个点
验证集: 45450 个点
测试集: 27270 个点

🚀 开始训练模型...
Epoch 39: 100%|██████████| 6815/6815 [02:20<00:00, 48.63it/s, v_num=12, train_loss=12.40, val_loss=22.80]

`Trainer.fit` stopped: `max_epochs=40` reached.


Epoch 39: 100%|██████████| 6815/6815 [02:20<00:00, 48.63it/s, v_num=12, train_loss=12.40, val_loss=22.80]


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU ava

Epoch 3:  29%|██▉       | 2317/7951 [1:40:30<4:04:22,  0.38it/s, v_num=10, train_loss=48.50, val_loss=34.00]

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU ava

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/

✅ gait 结果已保存至: ./forecast_results/gait_forecast.csv

📈 评估指标:
MAE: 3.0127
RMSE: 4.9801
MAPE: 301.1326
sMAPE: 42.5714
Directional_Accuracy: 0.8000
Threshold_Accuracy(>100): 1.0000
Runtime_Seconds: 5920.8514
📊 gait 可视化结果已保存


# Metro

In [ ]:
import os
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from darts import TimeSeries
from darts.models import TransformerModel
from darts.metrics import mae, mape
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from pytorch_lightning.loggers import CSVLogger

class UnifiedTransformerForecaster:
    def __init__(self, model_params: dict = None, default_config: dict = None):
        self.base_params = {
            'input_chunk_length': 36,
            'output_chunk_length': 12,
            'd_model': 64,
            'nhead': 4,
            'num_encoder_layers': 2,
            'num_decoder_layers': 2,
            'batch_size': 16,
            'n_epochs': 50,
            'pl_trainer_kwargs': {'accelerator': 'cpu', 'enable_progress_bar': False}
        }
        if model_params:
            self.base_params.update(model_params)

        # 提前定义 config
        self.config = {
            'results_dir': './forecast_results',
            'figure_dpi': 300,
            'train_ratio': 0.7,
            'val_ratio': 0.15
        }
        if default_config:
            self.config.update(default_config)

        # 设置 Logger
        from pytorch_lightning.loggers import CSVLogger
        self.base_params['pl_trainer_kwargs']['logger'] = CSVLogger(save_dir=self.config['results_dir'])

        # 初始化模型
        self.model = TransformerModel(**self.base_params)

        os.makedirs(self.config['results_dir'], exist_ok=True)

    def load_dataset(self, file_path: str, freq: str = None):
        df = pd.read_csv(file_path, encoding='utf-8', sep=',')
        dataset_name = os.path.basename(file_path).split('.')[0]
        # ———— 去重聚合：对相同 timestamp 取 y 的均值 ————
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = (
            df.sort_values('timestamp')
              .groupby('timestamp', as_index=False)['y']
              .mean()
        )

        # 生成完备索引并重建 DataFrame
        full_idx = self._generate_complete_index(df['timestamp'],freq)
        df = df.set_index('timestamp').reindex(full_idx).reset_index().rename(columns={'index':'timestamp'})
        df['y'] = df['y'].ffill()

        series = TimeSeries.from_dataframe(
            df,
            time_col='timestamp',
            value_cols='y',
            freq=freq or pd.infer_freq(full_idx) or 'h'
        )
        return dataset_name, series


    def _generate_complete_index(self, original_index, freq: str=None):
        """
        生成从 min 到 max 的完整时间索引。
        - 若提供 freq_override，则直接用它；
        - 否则尝试 pd.infer_freq 推断；
        - 推断失败时用最小正差值的 offset。
        """
        if len(original_index) < 2:
            return original_index
        
        if freq:
            freq_str = freq
        else:
            freq_str = pd.infer_freq(original_index) or ''
            if not freq_str:
                diffs    = pd.Series(original_index).diff().dropna()
                min_diff = diffs[diffs > pd.Timedelta(0)].min()
                freq_str = pd.tseries.frequencies.to_offset(min_diff).freqstr
       
        return pd.date_range(
            start=original_index.min(),
            end=original_index.max(),
            freq=freq_str
        )

    def train_and_predict(self, series: TimeSeries, dataset_name: str):
        try:
            total_len = len(series)
            train_size = int(total_len * self.config['train_ratio'])
            val_size = int(total_len * self.config['val_ratio'])
            train = series[:train_size]
            val = series[train_size:train_size + val_size]
            test = series[train_size + val_size:]

            print(f"\n📊 数据集划分:")
            print(f"训练集: {len(train)} 个点")
            print(f"验证集: {len(val)} 个点")
            print(f"测试集: {len(test)} 个点")

            print("\n🚀 开始训练模型...")
            start_time = time.time()
            self.model.fit(train, val_series=val, verbose=True)

            forecast = None
            history = train.append(val)

            for i in range(0, len(test), self.base_params['output_chunk_length']):
                current_input = history[-self.base_params['input_chunk_length']:]
                n_out = min(self.base_params['output_chunk_length'], len(test) - i)
                pred = self.model.predict(n=n_out, series=current_input)
                if pred is None or pred.values().size == 0:
                    print(f"❌ 第 {i} 步预测失败，终止滚动预测。")
                    break
                forecast = pred if forecast is None else forecast.append(pred)
                if i + self.base_params['output_chunk_length'] <= len(test):
                    history = history.append(test[i:i+self.base_params['output_chunk_length']])

            if forecast is None:
                print(f"❌ {dataset_name} 所有预测失败，跳过保存与绘图。")
                return

            runtime = time.time() - start_time
            self._save_results(train, test, forecast, dataset_name)

            y_true = test.values().flatten()
            y_pred = forecast.values().flatten()
            metrics = self.evaluate_all_metrics(y_true, y_pred, runtime=runtime)
            metrics_path = os.path.join("results/metric_csv/transformer", f"{dataset_name}_metrics.csv")
            pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
            print(f"📄 指标结果已保存至: {metrics_path}")
            print("\n📈 评估指标:")
            for k, v in metrics.items():
                print(f"{k}: {v:.4f}")

            self._plot_comparison(train, test, forecast, dataset_name)
            self._plot_loss_curve(dataset_name)

        except Exception as e:
            print(f"处理数据集 {dataset_name} 时出错: {str(e)}")

    def _plot_loss_curve(self, dataset_name):
        try:
            log_dir = self.model.trainer.logger.log_dir if self.model.trainer.logger else None
            if log_dir is None:
                print("⚠️ 无法获取 logger 日志目录")
                return

            log_path = os.path.join(log_dir, "metrics.csv")
            if not os.path.exists(log_path):
                print("⚠️ 未找到日志文件，无法绘制损失图")
                return

            df = pd.read_csv(log_path)

            # 检查是否存在列
            if 'epoch' not in df.columns:
                print("⚠️ metrics.csv 缺少 epoch 列")
                return

            train_loss = df[df['train_loss'].notna()][['epoch', 'train_loss']] if 'train_loss' in df.columns else pd.DataFrame()
            val_loss = df[df['val_loss'].notna()][['epoch', 'val_loss']] if 'val_loss' in df.columns else pd.DataFrame()

            plt.figure()
            if not train_loss.empty:
                plt.plot(train_loss['epoch'], train_loss['train_loss'], label="train_loss")
            if not val_loss.empty:
                plt.plot(val_loss['epoch'], val_loss['val_loss'], label="val_loss")

            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.legend()
            plt.title(f"{dataset_name} Loss Curve")
            plt.tight_layout()
            plt.savefig(os.path.join("results/loss_png/transformer", f"{dataset_name}_loss_curve.png"), dpi=self.config['figure_dpi'])

            plt.close()
        except Exception as e:
            print(f"绘图失败: {e}")

    def mean_absolute_percentage_error(self, y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

    def symmetric_mean_absolute_percentage_error(self, y_true, y_pred):
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10))

    def directional_accuracy(self, y_true, y_pred):
        return np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))

    def threshold_accuracy(self, y_true, y_pred, threshold=100):
        return accuracy_score((y_true > threshold).astype(int), (y_pred > threshold).astype(int))

    def evaluate_all_metrics(self, y_true, y_pred, threshold=100, runtime=None):
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAPE': self.mean_absolute_percentage_error(y_true, y_pred),
            'sMAPE': self.symmetric_mean_absolute_percentage_error(y_true, y_pred),
            'Directional_Accuracy': self.directional_accuracy(y_true, y_pred),
            f'Threshold_Accuracy(>{threshold})': self.threshold_accuracy(y_true, y_pred, threshold),
            'Runtime_Seconds': runtime if runtime is not None else -1
        }

    def _save_results(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        try:
            train_df = train.to_dataframe().rename(columns={train.components[0]: 'value'})
            test_df = test.to_dataframe().rename(columns={test.components[0]: 'value'})
            forecast_df = forecast.to_dataframe().rename(columns={forecast.components[0]: 'value'})
            train_df['type'] = 'train'
            test_df['type'] = 'test'
            forecast_df['type'] = 'forecast'
            result_df = pd.concat([train_df, test_df, forecast_df])
            if 'timestamp' not in result_df.columns:
                result_df = result_df.reset_index().rename(columns={'index': 'timestamp'})
            save_path = os.path.join(self.config['results_dir'], f"{dataset_name}_forecast.csv")
            result_df.to_csv(save_path, index=False)
            print(f"✅ {dataset_name} 结果已保存至: {save_path}")
        except Exception as e:
            print(f"保存结果时出错: {str(e)}")

    def _plot_comparison(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        plt.figure(figsize=(14, 7))
        test.plot(label="test", color='green', alpha=0.7)
        for i in range(0, len(forecast), self.base_params['output_chunk_length']):
            chunk = forecast[i:i+self.base_params['output_chunk_length']]
            chunk.plot(
                label="predict" if i == 0 else None,
                color='red',
                linestyle='--',
                marker='.',
                markersize=5
            )
        mae_score = mae(test, forecast)
        mape_score = mape(test, forecast)
        plt.title(f"{dataset_name} (MAE={mae_score:.2f}, MAPE={mape_score:.2f}%)")
        plt.axvline(test.start_time(), color='gray', linestyle='--', alpha=0.5)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        fig_path = os.path.join("results/prediction_png/transformer", f"{dataset_name}_comparison.png")

        plt.savefig(fig_path, dpi=self.config['figure_dpi'])
        plt.close()
        print(f"📊 {dataset_name} 可视化结果已保存")

if __name__ == "__main__":
    DATASET_PATHS = [
        './data/data_clean/metro.csv'
    ]
    MODEL_CONFIG = {
        'input_chunk_length': 36,
        'output_chunk_length': 12,
        'd_model': 64,
        'n_epochs': 30
    }
    forecaster = UnifiedTransformerForecaster(model_params=MODEL_CONFIG)
    for data_path in DATASET_PATHS:
        name, series = forecaster.load_dataset(data_path,freq='H')
        print(f"\n🔍 开始处理数据集: {name}")
        print(f"数据长度: {len(series)}")
        print(f"时间频率: {series.freq_str}")
        forecaster.train_and_predict(series, name)




/var/folders/25/xxhslgk16kq33dxhh4440t9r0000gn/T/ipykernel_19679/4271655947.py:92: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/darts/timeseries.py:5248: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  resampled_time_index = resampled_time_index.asfreq(freq)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightnin


🔍 开始处理数据集: metro
数据长度: 52551
时间频率: h

📊 数据集划分:
训练集: 36785 个点
验证集: 7882 个点
测试集: 7884 个点

🚀 开始训练模型...
Epoch 29: 100%|██████████| 2297/2297 [00:48<00:00, 47.79it/s, v_num=13, train_loss=3.02e+6, val_loss=6.45e+6]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 2297/2297 [00:48<00:00, 47.79it/s, v_num=13, train_loss=3.02e+6, val_loss=6.45e+6]


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU ava

✅ metro 结果已保存至: ./forecast_results/metro_forecast.csv

📈 评估指标:
MAE: 2130.7031
RMSE: 2490.3125
MAPE: 106.0223
sMAPE: 81.7088
Directional_Accuracy: 0.4026
Threshold_Accuracy(>100): 1.0000
Runtime_Seconds: 1475.1791
📊 metro 可视化结果已保存


# Productivity

In [8]:
import os
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from darts import TimeSeries
from darts.models import TransformerModel
from darts.metrics import mae, mape
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from pytorch_lightning.loggers import CSVLogger

class UnifiedTransformerForecaster:
    def __init__(self, model_params: dict = None, default_config: dict = None):
        self.base_params = {
            'input_chunk_length': 36,
            'output_chunk_length': 12,
            'd_model': 64,
            'nhead': 4,
            'num_encoder_layers': 2,
            'num_decoder_layers': 2,
            'batch_size': 16,
            'n_epochs': 50,
            'pl_trainer_kwargs': {'accelerator': 'cpu', 'enable_progress_bar': False}
        }
        if model_params:
            self.base_params.update(model_params)

        # 提前定义 config
        self.config = {
            'results_dir': './forecast_results',
            'figure_dpi': 300,
            'train_ratio': 0.7,
            'val_ratio': 0.15
        }
        if default_config:
            self.config.update(default_config)

        # 设置 Logger
        from pytorch_lightning.loggers import CSVLogger
        self.base_params['pl_trainer_kwargs']['logger'] = CSVLogger(save_dir=self.config['results_dir'])

        # 初始化模型
        self.model = TransformerModel(**self.base_params)

        os.makedirs(self.config['results_dir'], exist_ok=True)

    def load_dataset(self, file_path: str, freq: str = None):
        df = pd.read_csv(file_path, encoding='utf-8', sep=',')
        dataset_name = os.path.basename(file_path).split('.')[0]
        # ———— 去重聚合：对相同 timestamp 取 y 的均值 ————
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = (
            df.sort_values('timestamp')
              .groupby('timestamp', as_index=False)['y']
              .mean()
        )

        # 生成完备索引并重建 DataFrame
        full_idx = self._generate_complete_index(df['timestamp'],freq)
        df = df.set_index('timestamp').reindex(full_idx).reset_index().rename(columns={'index':'timestamp'})
        df['y'] = df['y'].ffill()

        series = TimeSeries.from_dataframe(
            df,
            time_col='timestamp',
            value_cols='y',
            freq=freq or pd.infer_freq(full_idx) or 'h'
        )
        return dataset_name, series


    def _generate_complete_index(self, original_index, freq: str=None):
        """
        生成从 min 到 max 的完整时间索引。
        - 若提供 freq_override，则直接用它；
        - 否则尝试 pd.infer_freq 推断；
        - 推断失败时用最小正差值的 offset。
        """
        if len(original_index) < 2:
            return original_index
        
        if freq:
            freq_str = freq
        else:
            freq_str = pd.infer_freq(original_index) or ''
            if not freq_str:
                diffs    = pd.Series(original_index).diff().dropna()
                min_diff = diffs[diffs > pd.Timedelta(0)].min()
                freq_str = pd.tseries.frequencies.to_offset(min_diff).freqstr
       
        return pd.date_range(
            start=original_index.min(),
            end=original_index.max(),
            freq=freq_str
        )

    def train_and_predict(self, series: TimeSeries, dataset_name: str):
        try:
            total_len = len(series)
            train_size = int(total_len * self.config['train_ratio'])
            val_size = int(total_len * self.config['val_ratio'])
            train = series[:train_size]
            val = series[train_size:train_size + val_size]
            test = series[train_size + val_size:]

            print(f"\n📊 数据集划分:")
            print(f"训练集: {len(train)} 个点")
            print(f"验证集: {len(val)} 个点")
            print(f"测试集: {len(test)} 个点")

            print("\n🚀 开始训练模型...")
            start_time = time.time()
            self.model.fit(train, val_series=val, verbose=True)

            forecast = None
            history = train.append(val)

            for i in range(0, len(test), self.base_params['output_chunk_length']):
                current_input = history[-self.base_params['input_chunk_length']:]
                n_out = min(self.base_params['output_chunk_length'], len(test) - i)
                pred = self.model.predict(n=n_out, series=current_input)
                if pred is None or pred.values().size == 0:
                    print(f"❌ 第 {i} 步预测失败，终止滚动预测。")
                    break
                forecast = pred if forecast is None else forecast.append(pred)
                if i + self.base_params['output_chunk_length'] <= len(test):
                    history = history.append(test[i:i+self.base_params['output_chunk_length']])

            if forecast is None:
                print(f"❌ {dataset_name} 所有预测失败，跳过保存与绘图。")
                return

            runtime = time.time() - start_time
            self._save_results(train, test, forecast, dataset_name)

            y_true = test.values().flatten()
            y_pred = forecast.values().flatten()
            metrics = self.evaluate_all_metrics(y_true, y_pred, runtime=runtime)
            metrics_path = os.path.join("results/metric_csv/transformer", f"{dataset_name}_metrics.csv")
            pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
            print(f"📄 指标结果已保存至: {metrics_path}")
            print("\n📈 评估指标:")
            for k, v in metrics.items():
                print(f"{k}: {v:.4f}")

            self._plot_comparison(train, test, forecast, dataset_name)
            self._plot_loss_curve(dataset_name)

        except Exception as e:
            print(f"处理数据集 {dataset_name} 时出错: {str(e)}")

    def _plot_loss_curve(self, dataset_name):
        try:
            log_dir = self.model.trainer.logger.log_dir if self.model.trainer.logger else None
            if log_dir is None:
                print("⚠️ 无法获取 logger 日志目录")
                return

            log_path = os.path.join(log_dir, "metrics.csv")
            if not os.path.exists(log_path):
                print("⚠️ 未找到日志文件，无法绘制损失图")
                return

            df = pd.read_csv(log_path)

            # 检查是否存在列
            if 'epoch' not in df.columns:
                print("⚠️ metrics.csv 缺少 epoch 列")
                return

            train_loss = df[df['train_loss'].notna()][['epoch', 'train_loss']] if 'train_loss' in df.columns else pd.DataFrame()
            val_loss = df[df['val_loss'].notna()][['epoch', 'val_loss']] if 'val_loss' in df.columns else pd.DataFrame()

            plt.figure()
            if not train_loss.empty:
                plt.plot(train_loss['epoch'], train_loss['train_loss'], label="train_loss")
            if not val_loss.empty:
                plt.plot(val_loss['epoch'], val_loss['val_loss'], label="val_loss")

            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.legend()
            plt.title(f"{dataset_name} Loss Curve")
            plt.tight_layout()
            plt.savefig(os.path.join("results/loss_png/transformer", f"{dataset_name}_loss_curve.png"), dpi=self.config['figure_dpi'])

            plt.close()
        except Exception as e:
            print(f"绘图失败: {e}")

    def mean_absolute_percentage_error(self, y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100

    def symmetric_mean_absolute_percentage_error(self, y_true, y_pred):
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-10))

    def directional_accuracy(self, y_true, y_pred):
        return np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred)))

    def threshold_accuracy(self, y_true, y_pred, threshold=100):
        return accuracy_score((y_true > threshold).astype(int), (y_pred > threshold).astype(int))

    def evaluate_all_metrics(self, y_true, y_pred, threshold=100, runtime=None):
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAPE': self.mean_absolute_percentage_error(y_true, y_pred),
            'sMAPE': self.symmetric_mean_absolute_percentage_error(y_true, y_pred),
            'Directional_Accuracy': self.directional_accuracy(y_true, y_pred),
            f'Threshold_Accuracy(>{threshold})': self.threshold_accuracy(y_true, y_pred, threshold),
            'Runtime_Seconds': runtime if runtime is not None else -1
        }

    def _save_results(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        try:
            train_df = train.to_dataframe().rename(columns={train.components[0]: 'value'})
            test_df = test.to_dataframe().rename(columns={test.components[0]: 'value'})
            forecast_df = forecast.to_dataframe().rename(columns={forecast.components[0]: 'value'})
            train_df['type'] = 'train'
            test_df['type'] = 'test'
            forecast_df['type'] = 'forecast'
            result_df = pd.concat([train_df, test_df, forecast_df])
            if 'timestamp' not in result_df.columns:
                result_df = result_df.reset_index().rename(columns={'index': 'timestamp'})
            save_path = os.path.join(self.config['results_dir'], f"{dataset_name}_forecast.csv")
            result_df.to_csv(save_path, index=False)
            print(f"✅ {dataset_name} 结果已保存至: {save_path}")
        except Exception as e:
            print(f"保存结果时出错: {str(e)}")

    def _plot_comparison(self, train: TimeSeries, test: TimeSeries, forecast: TimeSeries, dataset_name: str):
        plt.figure(figsize=(14, 7))
        test.plot(label="test", color='green', alpha=0.7)
        for i in range(0, len(forecast), self.base_params['output_chunk_length']):
            chunk = forecast[i:i+self.base_params['output_chunk_length']]
            chunk.plot(
                label="predict" if i == 0 else None,
                color='red',
                linestyle='--',
                marker='.',
                markersize=5
            )
        mae_score = mae(test, forecast)
        mape_score = mape(test, forecast)
        plt.title(f"{dataset_name} (MAE={mae_score:.2f}, MAPE={mape_score:.2f}%)")
        plt.axvline(test.start_time(), color='gray', linestyle='--', alpha=0.5)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        fig_path = os.path.join("results/prediction_png/transformer", f"{dataset_name}_comparison.png")

        plt.savefig(fig_path, dpi=self.config['figure_dpi'])
        plt.close()
        print(f"📊 {dataset_name} 可视化结果已保存")

if __name__ == "__main__":
    DATASET_PATHS = [
        './data/data_clean/productivity_processed.csv'
    ]
    MODEL_CONFIG = {
        'input_chunk_length': 36,
        'output_chunk_length': 12,
        'd_model': 64,
        'n_epochs': 50
    }
    forecaster = UnifiedTransformerForecaster(model_params=MODEL_CONFIG)
    for data_path in DATASET_PATHS:
        name, series = forecaster.load_dataset(data_path,freq='H')
        print(f"\n🔍 开始处理数据集: {name}")
        print(f"数据长度: {len(series)}")
        print(f"时间频率: {series.freq_str}")
        forecaster.train_and_predict(series, name)




/var/folders/25/xxhslgk16kq33dxhh4440t9r0000gn/T/ipykernel_19679/227898254.py:92: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/darts/timeseries.py:5248: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  resampled_time_index = resampled_time_index.asfreq(freq)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning


🔍 开始处理数据集: productivity_processed
数据长度: 1657
时间频率: h

📊 数据集划分:
训练集: 1159 个点
验证集: 248 个点
测试集: 250 个点

🚀 开始训练模型...
Epoch 49: 100%|██████████| 70/70 [00:01<00:00, 42.57it/s, v_num=8, train_loss=0.000174, val_loss=0.00044] 

`Trainer.fit` stopped: `max_epochs=50` reached.


Epoch 49: 100%|██████████| 70/70 [00:01<00:00, 42.55it/s, v_num=8, train_loss=0.000174, val_loss=0.00044]


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
GPU available: True (mps), used: False
TPU ava

✅ productivity_processed 结果已保存至: ./forecast_results/productivity_processed_forecast.csv

📈 评估指标:
MAE: 0.0092
RMSE: 0.0119
MAPE: 1.2860
sMAPE: 1.2856
Directional_Accuracy: 0.4859
Threshold_Accuracy(>100): 1.0000
Runtime_Seconds: 71.1065
📊 productivity_processed 可视化结果已保存
